
# Unit 2: Symbolic Integration

In the last notebook, we learned that the Fundamental Theorem of Calculus (FTC) gives us a shortcut. If we can find the antiderivative, $F(x)$, we can find the exact area under a curve from $a$ to $b$ by simply calculating:


$$\int_a^b f(x)dx = F(b) - F(a)$$

But finding the antiderivative by hand is tedious. Today, we are going to resurrect the Object-Oriented symbolic engine we built in Unit 1, and we are going to teach it how to integrate.

*(Note: Building a complete symbolic integrator that handles the Reverse Chain Rule and U-Substitution is incredibly difficult. We are going to stick to polynomials, which power 90% of basic physics and engineering problems!)*


In [ ]:

# Let's set up our base class again, adding the .integrate() method
class Expression:
    def evaluate(self, context):
        pass
        
    def derive(self, var_name):
        pass
        
    def integrate(self, var_name):
        raise NotImplementedError("Each subclass must implement integrate()")
        
    def __add__(self, other):
        return Add(self, other)
        
    def __mul__(self, other):
        return Multiply(self, other)



### **[Markdown Cell]**

## 1. Integrating Constants

The derivative of $3x$ is $3$. Therefore, if we move in reverse, the integral of $3$ with respect to $x$ must be $3x$.

Mathematically: $\int k dx = kx$

Let's build this logic into our `Constant` class.


In [ ]:
class Constant(Expression):
    def __init__(self, value):
        self.value = value
        
    def __str__(self): return str(self.value)
    
    def evaluate(self, context): return self.value
    
    def integrate(self, var_name):
        # The integral of a constant k is (k * x)
        return Multiply(self, Variable(var_name))

class Variable(Expression):
    def __init__(self, name):
        self.name = name
        
    def __str__(self): return self.name
    
    def evaluate(self, context): return context[self.name]


## 2. The Reverse Power Rule

To take a derivative using the power rule, you multiply by the exponent, and subtract 1 from the exponent.
To integrate, you do the exact opposite: **Add 1 to the exponent, and divide by the new exponent.**

$$\int x^n dx = \frac{1}{n+1} x^{n+1}$$

Let's build a `Power` class that knows how to integrate itself. *(For this simple engine, we will assume the base is our variable and the exponent is a constant).*


In [ ]:
class Power(Expression):
    def __init__(self, base, exponent):
        self.base = base
        self.exponent = exponent
        
    def __str__(self): return f"({self.base}^{self.exponent})"
    
    def evaluate(self, context):
        return self.base.evaluate(context) ** self.exponent.evaluate(context)
        
    def integrate(self, var_name):
        # Only works if we are integrating with respect to our base variable
        if isinstance(self.base, Variable) and self.base.name == var_name:
            # We need the current exponent value
            n = self.exponent.value
            
            if n == -1:
                raise ValueError("Integral of x^-1 is ln|x|, which is not supported in this basic engine!")
            
            # 1 / (n + 1)
            new_coefficient = Constant(1 / (n + 1))
            # x^(n + 1)
            new_power = Power(self.base, Constant(n + 1))
            
            return Multiply(new_coefficient, new_power)
        else:
            raise NotImplementedError("Complex integration not supported.")


## 3. The Sum Rule for Integration

Just like derivatives, the integral of $f(x) + g(x)$ is simply the integral of $f(x)$ plus the integral of $g(x)$. Our `Add` class just passes the command down to its children!


In [ ]:
class Add(Expression):
    def __init__(self, left, right):
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def __str__(self): return f"({self.left} + {self.right})"
    
    def evaluate(self, context):
        return self.left.evaluate(context) + self.right.evaluate(context)
        
    def integrate(self, var_name):
        # Integrate the left, integrate the right, add them together!
        return Add(self.left.integrate(var_name), self.right.integrate(var_name))
        
# We also need a basic Multiply class for coefficients like 3 * x^2
class Multiply(Expression):
    def __init__(self, left, right):
        self.left = left if isinstance(left, Expression) else Constant(left)
        self.right = right if isinstance(right, Expression) else Constant(right)
        
    def __str__(self): return f"({self.left} * {self.right})"
    
    def evaluate(self, context):
        return self.left.evaluate(context) * self.right.evaluate(context)
        
    def integrate(self, var_name):
        # We will only support integrating (Constant * Expression) to keep it simple
        if isinstance(self.left, Constant):
            return Multiply(self.left, self.right.integrate(var_name))
        else:
            raise NotImplementedError("Product rule integration (Integration by Parts) not supported here.")



## Challenge: Evaluating the Definite Integral

You have built the pieces. Now it is time to use them to solve a real calculus problem perfectly, with no rectangles and no estimation.

**The Problem:**
Find the exact area under the curve $f(x) = 3x^2 + 4$ from $x=1$ to $x=3$.

**Your Task:**

1. Use your classes to build the mathematical expression: $3x^2 + 4$. *(Hint: `Multiply(Constant(3), Power(Variable('x'), Constant(2))) + Constant(4)`)*
2. Print your original expression to make sure it looks right.
3. Call `.integrate('x')` on your expression to generate the antiderivative $F(x)$, and print it.
4. Use the `.evaluate()` method on your new antiderivative twice: once with `{'x': 3}` and once with `{'x': 1}`.
5. Subtract the two values (FTC Part 2!) and print the final, exact area.


In [ ]:
# 1. Build the equation here:

# 2. Integrate it:

# 3. Evaluate F(b) - F(a):

